## Part f Classification

In [34]:
from pathlib import Path
import sys


here = Path.cwd()
candidates = [here] + list(here.parents)
for p in candidates:
    if (p / "Code").is_dir():
        sys.path.insert(0, str(p))
        break
else:
    raise RuntimeError("Couldn't find a 'Code' folder in this project.")


import autograd.numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns 
import argparse
import random
import itertools

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

from Code.ffnn2 import NeuralNetwork as FFNN 
from Code.scheduler import Adam, RMS_prop, Constant, Scheduler
from Code.activations import sigmoid, identity, derivate, RELU, LRELU
from Code.architectures import build_architectures, build_architectures_class, build_architectures_2
from Code.comparison import run_classification_comparison
from Code.cost import CostCrossEntropy, dCostCrossEntropy, softmax





In [35]:
"""Data loading and preprocessing"""

mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')

# Features: float32 in [0,1]
X = mnist.data.astype(np.float32) / 255.0

# Labels: keep as integers (OpenML may give strings -> cast to int64)
y = mnist.target.astype(np.int64)

# Train/test split (stratify to keep class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Ensure dtypes (features float32; labels int64)
X_train = X_train.astype(np.float32)
X_test  = X_test.astype(np.float32)
y_train = y_train.astype(np.int64).ravel()
y_test  = y_test.astype(np.int64).ravel()

# One-hot encode labels (indices must be integers)
n_classes = int(np.max(y_train)) + 1  # for MNIST this is 10
Y_train = np.eye(n_classes, dtype=np.float32)[y_train]
Y_test  = np.eye(n_classes, dtype=np.float32)[y_test]



In [36]:

#drop GD and SGD for speed
optimizers_stage1 = {
    "rms_prop": (RMS_prop, {"rho": 0.9}),
    "adam":     (Adam,     {"rho": 0.9, "rho2": 0.999}),
}

# drop sigmoid for speed 
activations_stage1 = {
    "relu": RELU,
    "lrelu": LRELU,
}

N = 10
depths = (1, 2)
widths = (64, 128)
input_nodes = X_train.shape[1]

architectures_to_sweep = build_architectures_2(depths=depths, widths=widths, out_dim=N)

In [37]:
def run_classification(
    optimizer_class, layer_output_sizes, hidden_activation_func,
    X_train, Y_train, X_test, Y_test,
    eta, lam, epochs=10, batches=64, **opt_kwargs
):
    # 1. Define Activation Functions for all layers
    n_hidden_layers = len(layer_output_sizes) - 1
    # Hidden layers use the chosen function; Output layer MUST use softmax for multiclass CE [2, 6]
    activation_funcs = [hidden_activation_func] * n_hidden_layers + [softmax]
    activation_ders = [derivate(f) for f in activation_funcs]

    nn = FFNN(
        network_input_size=input_nodes,
                    layer_output_sizes=tuple(layer_output_sizes),
                    activation_funcs=activation_funcs,
                    activation_ders=activation_ders,
                    cost_fun=CostCrossEntropy,
                    cost_der=dCostCrossEntropy,
                    seed=42
    )
    nn.reset_weights()

    # 3. Instantiate Scheduler (e.g., Adam, RMSprop, Constant)
    scheduler_instance = optimizer_class(eta=eta, **opt_kwargs)

    # 4. Train the network (using L2 regularization 'lam')
    nn.fit(
        X=X_train,
        t=Y_train, # Expects one-hot encoded targets [14]
        scheduler=scheduler_instance,
        epochs=epochs,
        batches=batches,
        lam=lam # L2 regularization [15]
    )

    # 5. Evaluate Performance (Accuracy is the metric for classification [16])
    # The FFNN predict method typically returns probabilities (after Softmax)
    test_probs = nn.predict(X_test)
    y_hat = np.argmax(test_probs, axis=1) # Get predicted labels (0-9) [17]
    
    # Convert true one-hot labels back to integer labels for comparison
    y_true_labels = np.argmax(Y_test, axis=1) 
    
    accuracy = np.mean(y_hat == y_true_labels) # Equivalent to accuracy_score [16]
    
    return accuracy

## Multi-stage tuning

Stage one - optimizer and learning rate sweep

In [38]:
eta_vals = np.logspace(-5, -1, 5) #logspace for speed 
lambda_vals = [0.0] #iterate over L1 and L2 later
epochs_sweep = 1 # Keep epochs low initially to speed up the sweep
batch_size = 512

all_combinations = list(itertools.product(
    architectures_to_sweep.items(),          # Architecture (name, layer_sizes)
    optimizers_stage1.items(),             # Optimizer (name, (class, params))
    activations_stage1.items(),     # Activation (name, func)
    eta_vals,
    lambda_vals
))
print(f"Total combos: {len(all_combinations)}")

#set a maximum number of combinations to run
N_max_combos = 250
if len(all_combinations) > N_max_combos:
    random.seed(42)
    search_combos = random.sample(all_combinations, k=N_max_combos)
    print(f"Running randomized search on {N_max_combos} samples.")
else:
    search_combos = all_combinations

Total combos: 80


In [ ]:
results = []
for combo in search_combos:
    (arch_name, layer_sizes), (opt_name, (opt_class, fixed_params)), \
    (act_name, act_func), eta, lam = combo
    
    # Consolidate kwargs for the optimizer/scheduler
    opt_kwargs = {'eta': eta} 
    opt_kwargs.update(fixed_params)

    # Run the model
    try:
        accuracy = run_classification(
            opt_class, layer_sizes, act_func,
            X_train, Y_train, X_test, Y_test,
            epochs=epochs_sweep, batches=batch_size,
            # Explicitly pass regularization term to the NN training
            lam=lam, 
            **opt_kwargs
        )

        results.append({
            'architecture': arch_name,
            'optimizer': opt_name,
            'activation': act_name,
            'eta': eta,
            'lambda': lam,
            'accuracy': accuracy
        })
        print(f"Running {arch_name}/{opt_name}/{act_name} (eta={eta:.1e}): Accuracy={accuracy:.4f}")
        
    except Exception as e:
        # Handle failures (e.g., exploding gradients) gracefully
        print(f"Skipped {arch_name}/{opt_name}. Error: {e}")
        results.append({
            'architecture': arch_name,
            'optimizer': opt_name,
            'activation': act_name,
            'eta': eta,
            'lambda': lam,
            'accuracy': np.nan 
        })


Running 1L-64N/rms_prop/relu (eta=1.0e-05): Accuracy=0.6847
Running 1L-64N/rms_prop/relu (eta=1.0e-04): Accuracy=0.7700
Running 1L-64N/rms_prop/relu (eta=1.0e-03): Accuracy=0.9137
Running 1L-64N/rms_prop/relu (eta=1.0e-02): Accuracy=0.9451
Running 1L-64N/rms_prop/relu (eta=1.0e-01): Accuracy=0.8765
Running 1L-64N/rms_prop/lrelu (eta=1.0e-05): Accuracy=0.6847
Running 1L-64N/rms_prop/lrelu (eta=1.0e-04): Accuracy=0.7694
Running 1L-64N/rms_prop/lrelu (eta=1.0e-03): Accuracy=0.9141
Running 1L-64N/rms_prop/lrelu (eta=1.0e-02): Accuracy=0.9481
Running 1L-64N/rms_prop/lrelu (eta=1.0e-01): Accuracy=0.8890
Running 1L-64N/adam/relu (eta=1.0e-05): Accuracy=0.6911
Running 1L-64N/adam/relu (eta=1.0e-04): Accuracy=0.7747
Running 1L-64N/adam/relu (eta=1.0e-03): Accuracy=0.9078
Running 1L-64N/adam/relu (eta=1.0e-02): Accuracy=0.9548
Running 1L-64N/adam/relu (eta=1.0e-01): Accuracy=0.2685
Running 1L-64N/adam/lrelu (eta=1.0e-05): Accuracy=0.6922
Running 1L-64N/adam/lrelu (eta=1.0e-04): Accuracy=0.7728


In [ ]:
df_results = pd.DataFrame(results)

# Find the best overall configuration
best_result = df_results.loc[df_results['accuracy'].idxmax()]
print("\n--- Best Configuration Found ---")
print(best_result)

# Example: Pivot and plot the results for a fixed architecture/optimizer/activation pair
# This helps generate the 2D heatmaps requested in the analysis [20, 21]
pivot_table = df_results[
    (df_results['architecture'] == '2L-128N') & 
    (df_results['optimizer'] == 'Adam') & 
    (df_results['activation'] == 'ReLU')
].pivot_table(index='eta', columns='lambda', values='accuracy')